In [ ]:
# algorithm outline
# 1. generate data: 
    # input: 
    #   calculate the concetrations of 3 drone from a random source
    #   calculate the concentrations of 3 drones from (0, 0, 0)
    #   find dc for all drones (random source - (0, 0, 0))

    # loss:
    # distance
    # -dx, -dy, -dz should equal source

    # output: 
    #   dx, dy, dz (translation from (0, 0, 0) to random source point)

In [5]:
# imports
import numpy as np

import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [112]:
# # generate input

Q = 500     # Stronger source
u = 0.7      # Wind speed
H = 10       # Height breaks z symmetry

ay, by = 0.5, 0.8 # Lateral spread
az, bz = 1, 0.8

def plume_function(x0, y0, z0, x, y, z):
    dx = max(x - x0, 0.0001)
    dy = max(y - y0, 0.0001)
    dz = max(z - z0, 0.0001)

    s_y = ay * dx**by
    s_z = az * dx**bz
    s_y = max(s_y, 0.00002)
    s_z = max(s_z, 0.00002)

    term1 = Q / (2 * np.pi * s_y * s_z * u)
    term2 = np.exp(-dy**2 / (2 * s_y**2))
    term3 = np.exp(-(dz - H)**2 / (2 * s_z**2))
    c = term1 * term2 * term3
    return c



In [113]:
# create bit map
source = [0, 25, 25]
bit_map = [[[0 for _ in range(50)] for _ in range(50)] for _ in range(50)]

for i in range(0, 50):
    for j in range(0, 50):
        for k in range(0, 50):
            bit_map[i][j][k] = plume_function(source[0], source[1], source[2], i, j, k)

In [37]:
# bit_map

In [38]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Create a sample 3D array
# array_3d = np.array(bit_map) # use 5x5x5 to keep graph readable

# # Flatten to 1D
# flattened = array_3d.flatten()

# # Plot as 1D bar graph
# plt.figure(figsize=(10, 4))
# plt.bar(range(len(flattened)), flattened)
# plt.title("Bar Graph of Flattened 3D Array")
# plt.xlabel("Index")
# plt.ylabel("Value")
# plt.show()


In [136]:
epsilon = 0.0000000001
def find_conc(bit_map, len_map, conc):
    found = []
    for i in range(len_map):
        for j in range(len_map):
            for k in range(len_map):
                if(bit_map[i][j][k] <= conc + epsilon and conc - epsilon <= bit_map[i][j][k]):
                    found.append([i, j, k])
    return found

def conc_match(bit_map, conc, i, j, k, shift):
    new_i = i+ shift
    new_j = j+shift
    new_k = k + shift
    return (bit_map[new_i][new_j][new_k] <= conc + epsilon and bit_map[new_i][new_j][new_k] >= conc - epsilon)

def get_source(bit_map, concentrations):
    c1, c2, c3 = concentrations
    possible = find_conc(bit_map=bit_map, len_map=47, conc=c1)
    print(possible)

    c2_works = []
    # c2_works = find_conc(bit_map=bit_map, len_map=50, conc=c2)
    print(possible)
    for i, j, k in possible:
        if(conc_match(bit_map=bit_map, conc=c2, i=i, j=j, k=k, shift=2)):
            c2_works.append([i, j, k])
    
    print(c2_works)

    final = []
    for i, j, k in c2_works:
        if(conc_match(bit_map=bit_map, conc=c3, i=i, j=j, k=k, shift=3)):
            final.append([i, j, k])
    print(final)
    
    if(len(final) == 0):
        final = c2_works
    
    if(len(final) == 0):
        final = possible
    dx, dy, dz = final[0]
    return dx, dy, dz



In [137]:
x0, y0, z0 = np.random.randint(0, 50, 3)

drone_first = np.random.randint(0, 46, 3)
drone_points = [
    drone_first, 
    [drone_first[0]+2, drone_first[1]+2, drone_first[2]+2], 
    [drone_first[0]+3, drone_first[1]+3, drone_first[2]+3]
]

print(drone_points)
concentrations = [
    plume_function(x0, y0, z0, drone_points[0][0], drone_points[0][1], drone_points[0][2]),
    plume_function(x0, y0, z0, drone_points[2][0], drone_points[2][1], drone_points[2][2]),
    plume_function(x0, y0, z0, drone_points[1][0], drone_points[1][1], drone_points[1][2])
]

print("concentrations: ", concentrations)
print("dx, dy, dz", get_source(bit_map, concentrations))
print("drone_1: ", drone_first)
print("source position: ", x0, y0, z0)
print("concentration at source: ", bit_map[x0][y0][z0], bit_map[x0 + 2][y0 + 2][z0 + 2])



[array([44, 45, 11]), [np.int64(46), np.int64(47), np.int64(13)], [np.int64(47), np.int64(48), np.int64(14)]]
concentrations:  [np.float64(0.0), np.float64(3.6806099572375e-07), np.float64(5.940778018758708e-21)]
[[0, 0, 0], [0, 0, 1], [0, 0, 2], [0, 0, 3], [0, 0, 4], [0, 0, 5], [0, 0, 6], [0, 0, 7], [0, 0, 8], [0, 0, 9], [0, 0, 10], [0, 0, 11], [0, 0, 12], [0, 0, 13], [0, 0, 14], [0, 0, 15], [0, 0, 16], [0, 0, 17], [0, 0, 18], [0, 0, 19], [0, 0, 20], [0, 0, 21], [0, 0, 22], [0, 0, 23], [0, 0, 24], [0, 0, 25], [0, 0, 26], [0, 0, 27], [0, 0, 28], [0, 0, 29], [0, 0, 30], [0, 0, 31], [0, 0, 32], [0, 0, 33], [0, 0, 34], [0, 0, 36], [0, 0, 37], [0, 0, 38], [0, 0, 39], [0, 0, 40], [0, 0, 41], [0, 0, 42], [0, 0, 43], [0, 0, 44], [0, 0, 45], [0, 0, 46], [0, 1, 0], [0, 1, 1], [0, 1, 2], [0, 1, 3], [0, 1, 4], [0, 1, 5], [0, 1, 6], [0, 1, 7], [0, 1, 8], [0, 1, 9], [0, 1, 10], [0, 1, 11], [0, 1, 12], [0, 1, 13], [0, 1, 14], [0, 1, 15], [0, 1, 16], [0, 1, 17], [0, 1, 18], [0, 1, 19], [0, 1, 20], [0